### 第六周练习：价格合适！（Price is Right）

## 练习目标

本笔记本演示如何用 **OpenAI 微调（Fine-Tuning）** 训练一个 `gpt-4.1-nano` 价格预测器：

- 数据：Ed Donner 整理的亚马逊抓取子集（`ed-donner/items_lite` / `items_full`）
- 流程：取小样本 → 做成 JSONL → 上传 → 创建微调任务 → 用微调模型推理比价

对应课程第 6 周「用微调做定价」的社区最短路径版。

## 怎么跑

1. 准备好 `.env` 中的 OpenAI / Hugging Face 凭证（见下一格）
2. 确保本地有 `pricer.items.Item`（课程配套包）
3. 从上到下依次运行；微调任务在云端排队，需轮询 job 状态


In [ ]:
# ========== 导入 + 环境：OpenAI / HF 登录，并选择 lite 数据集开关 ==========

# 标准库：读环境变量、路径、JSON
import os
import sys
import json
# dotenv：把 .env 密钥灌进环境变量
from dotenv import load_dotenv
# Hugging Face Hub 登录（拉数据集可能需要）
from huggingface_hub import login
# OpenAI 官方客户端：files / fine_tuning / chat
from openai import OpenAI

# 课程配套：Item 封装了 from_hub 与 summary/price 字段
from pricer.items  import Item

# --- 环境 ---
# True=小数据集 items_lite；False=全量 items_full（更慢更大）
LITE_MODE = True
# override=True：.env 覆盖已有环境变量
load_dotenv(override=True)
# 读取 HF token（键名必须是 HF_TOKEN）
hf_token = os.environ['HF_TOKEN']
# 登录 Hub；同时写入 git credential 方便后续
login(hf_token, add_to_git_credential=True)

# 默认从环境变量 OPENAI_API_KEY 创建客户端
openai = OpenAI()


## 加载数据集

按上一格的 `LITE_MODE` 选择 `ed-donner/items_lite` 或更大的 `ed-donner/items_full`，并拆成 train / val / test。


In [ ]:
# ========== 从 Hub 拉取 Item 列表：train / val / test ==========

# 三元表达式：lite 或 full 的数据集 id（字符串保持原样）
dataset = "ed-donner/items_lite" if LITE_MODE else "ed-donner/items_full"
# Item.from_hub：课程封装，返回三个 list-like 集合
train, val, test = Item.from_hub(dataset)

# 打印规模，确认下载成功
print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")


## 小样本微调子集

OpenAI 微调建议先用约 **100** 条训练 + **100** 条验证摸清流程与费用；全量再放大。


In [ ]:
# ========== 截取前 100 条作为微调 train / validation ==========

# 切片：训练子集
fine_tune_train = train[:100]
# 切片：验证子集（用于微调任务的 validation_file）
fine_tune_validation = val[:100]


## 准备 JSONL

把每条商品变成 Chat 消息对，再序列化成 **JSON Lines**，以便 `openai.files.create(..., purpose="fine-tune")` 上传。


In [ ]:
# ========== messages_for：一条 Item → user 问价 + assistant 答美元价格 ==========

def messages_for(item):
    # user 内容：要求只回报价、不要解释（英文 prompt 保持原样）
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [
        {"role": "user", "content": message},
        # assistant 目标：格式化成 $xx.xx
        {"role": "assistant", "content": f"${item.price:.2f}"}
    ]


## 拼出 JSONL 字符串

把多条 `messages` 包进 `{"messages": [...]}`，每行一条，供写入 `.jsonl` 文件。


In [ ]:
# ========== make_jsonl：把 Item 列表转成多行 JSONL 文本 ==========

def make_jsonl(items):
    # 累积字符串（最后 strip 去掉尾部多余换行）
    result = ""
    for item in items:
        # 得到 [{user},{assistant}]
        messages = messages_for(item)
        # messages 数组本身先 dumps
        messages_str = json.dumps(messages)
        # 包进 OpenAI 微调要求的外层 messages 字段
        result += '{"messages": ' + messages_str +'}\n'
    return result.strip()


In [ ]:
# ========== 冒烟检查：打印前 8 条训练样本的 JSONL ==========

# 肉眼确认 summary / 价格格式是否合理
print(make_jsonl(train[:8]))


## 写入本地 jsonl 文件

分别写出训练与验证文件，路径相对笔记本工作目录下的 `jsonl/`。


In [ ]:
# ========== write_jsonl：调用 make_jsonl 并写入指定文件名 ==========

def write_jsonl(items, filename):
    # 文本模式写入（utf-8 默认）
    with open(filename, "w") as f:
        jsonl = make_jsonl(items)
        f.write(jsonl)


In [ ]:
# ========== 写出微调训练集 JSONL ==========

# 路径需事先存在 jsonl/ 目录（或自行 mkdir）
write_jsonl(fine_tune_train, "jsonl/fine_tune_train.jsonl")


In [ ]:
# ========== 写出微调验证集 JSONL ==========

write_jsonl(fine_tune_validation, "jsonl/fine_tune_validation.jsonl")


## 上传到 OpenAI

以二进制打开 jsonl，用 `purpose="fine-tune"` 创建 File 对象，供下一步 `fine_tuning.jobs.create` 引用。


In [ ]:
# ========== 上传训练文件到 OpenAI Files API ==========

# rb：按文件原样上传
with open("jsonl/fine_tune_train.jsonl", "rb") as f:
    train_file = openai.files.create(file=f, purpose="fine-tune")


In [ ]:
# 在笔记本里直接显示 File 对象（含 id、bytes、status 等）
train_file


In [ ]:
# ========== 上传验证文件到 OpenAI Files API ==========

with open("jsonl/fine_tune_validation.jsonl", "rb") as f:
    validation_file = openai.files.create(file=f, purpose="fine-tune")


In [ ]:
# 显示验证 File 对象，确认拿到 validation_file.id
validation_file


## 创建微调任务

指定基座模型、训练/验证文件、种子与少量超参；`suffix` 会出现在最终微调模型名里。


In [ ]:
# ========== 提交 fine-tuning job（云端排队训练） ==========

openai.fine_tuning.jobs.create(
    # 训练文件 id
    training_file=train_file.id,
    # 验证文件 id
    validation_file=validation_file.id,
    # 基座模型 id（保持原样）
    model="gpt-4.1-nano-2025-04-14",
    # 可复现种子
    seed=42,
    # 小规模试跑：1 epoch、batch=1
    hyperparameters={"n_epochs": 1, "batch_size": 1},
    # 模型名后缀，便于在仪表盘识别
    suffix="pricer"
)


In [ ]:
# 列出最近 1 个微调任务，快速看状态
openai.fine_tuning.jobs.list(limit=1)


In [ ]:
# ========== 取出最新一条 job 的 id，后面 retrieve / list_events 都用它 ==========

job_id = openai.fine_tuning.jobs.list(limit=1).data[0].id


In [ ]:
# 显示 job_id 字符串
job_id


In [ ]:
# 按 id 拉取任务详情（status、fine_tuned_model 等）
openai.fine_tuning.jobs.retrieve(job_id)


In [ ]:
# ========== 查看最近事件日志（训练进度 / 报错线索） ==========

openai.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id, limit=10).data


## 测试微调模型

任务完成后，从 job 上读取 `fine_tuned_model` 名称，再用同样的问价 prompt 做推理。


In [ ]:
# ========== 从已完成 job 读取微调后的模型名 ==========

fine_tuned_model_name = openai.fine_tuning.jobs.retrieve(job_id).fine_tuned_model


In [ ]:
# 显示模型名（形如 ft:gpt-4.1-nano-...:pricer:...）
fine_tuned_model_name


In [ ]:
# ========== 推理用 messages：只要 user，不要把真值价格放进上下文 ==========

def test_messages_for(item):
    # 与训练时 user 文案一致（保持英文）
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [
        {"role": "user", "content": message},
    ]


In [ ]:
# 用测试集第一条预览 messages 结构
test_messages_for(test[0])


## 推理函数

封装一次 `chat.completions.create`：模型用微调名，限制 `max_tokens` 只要短价格串。


In [ ]:
# ========== gpt_4__1_nano_fine_tuned：对单个 Item 调微调模型报价 ==========

def gpt_4__1_nano_fine_tuned(item):
    response = openai.chat.completions.create(
        # 使用上一格拿到的 ft: 模型名
        model=fine_tuned_model_name,
        # 仅 user 消息
        messages=test_messages_for(item),
        # 价格短答，限制生成长度
        max_tokens=7
    )
    # 取出助手文本
    return response.choices[0].message.content


In [ ]:
# ========== 单条对照：打印真值价格 vs 微调模型预测 ==========

print(test[0].price)
print(gpt_4__1_nano_fine_tuned(test[0]))


### 作者记录：`ft:gpt-4.1-nano-2025-04-14:personal:pricer:DVWc36rw` 的结果

- 误差（Error）：$74.48
- MSE：19,505
- R²：11.3%

（以上为贡献者跑完后的度量摘录；你本地重新微调会得到不同的 `ft:` 模型名与分数。）
